f > f bag > z

In [3]:
import os
# GPU 0번 강제 할당
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import glob
import random
import torch

from models.network import MonoCMIL

# 1. 실제 데이터 경로 세팅
base_path = '/data3/jinsol/TCGA/features_conch_256/pt_files'
pt_files = glob.glob(os.path.join(base_path, '*/*.pt'))

if not pt_files:
    print(f"에러: 해당 경로에서 .pt 파일을 찾을 수 없어. ({base_path})")
else:
    sample_path = random.choice(pt_files)
    print(f"✅ 테스트 파일: {sample_path}\n")

    # 2. 피처 로드
    features = torch.load(sample_path)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    features = features.to(device)
    num_patches, feature_dim = features.shape

    # 3. 모델 초기화
    model = MonoCMIL(input_dim=feature_dim, hidden_dim=512, z_dim=256).to(device)
    model.eval()

    # 4. 모델 통과 및 실제 값 샘플 출력
    with torch.no_grad():
        z, f_bag, A = model(features)

        print("[STEP 1] 원본 패치 피처 샘플")
        print(f"  ▶ 전체 형태: {features.shape}")
        # 첫 번째 패치의 앞 5개 특징값 출력
        print(f"  ▶ 첫 패치의 앞 5개 값: {features[0, :5].cpu().numpy()}\n")
        print("-" * 60)

        print("[STEP 2] 생성된 Feature Bag 벡터 (f_bag) 샘플")
        print(f"  ▶ 전체 형태: {f_bag.shape}")
        # 512차원 중 앞 5개 값 출력
        print(f"  ▶ f_bag 앞 5개 값: {f_bag[0, :5].cpu().numpy()}\n")
        print("-" * 60)

        print("[STEP 3] 최종 잠재 벡터 (z) 샘플")
        print(f"  ▶ 전체 형태: {z.shape}")
        # 256차원 중 앞 5개 값 출력
        print(f"  ▶ z 앞 5개 값: {z[0, :5].cpu().numpy()}\n")
        print("=" * 60)
        print("🎉 벡터 값 추출 및 확인 성공!")

✅ 테스트 파일: /data3/jinsol/TCGA/features_conch_256/pt_files/TCGA-BRCA/TCGA-A8-A09R-01A-01-BS1.e7c57b70-0da9-4500-adf7-e58735f6aac7.pt

[STEP 1] 원본 패치 피처 샘플
  ▶ 전체 형태: torch.Size([29860, 512])
  ▶ 첫 패치의 앞 5개 값: [ 0.18316671 -0.3543522  -0.4127538   1.0131879  -0.70534754]

------------------------------------------------------------
[STEP 2] 생성된 Feature Bag 벡터 (f_bag) 샘플
  ▶ 전체 형태: torch.Size([1, 512])
  ▶ f_bag 앞 5개 값: [3.8267663e-01 5.4388046e-02 5.7207223e-04 3.0866086e-01 7.2815973e-01]

------------------------------------------------------------
[STEP 3] 최종 잠재 벡터 (z) 샘플
  ▶ 전체 형태: torch.Size([1, 256])
  ▶ z 앞 5개 값: [ 0.09921794  0.08345747 -0.24124122  0.07494768  0.09494659]

🎉 벡터 값 추출 및 확인 성공!


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import glob
import torch
from torch.utils.data import DataLoader

# 저장한 실제 모듈들 임포트
from data.dataset import TCGABags
from models.network import MonoCMIL
from core.losses import HypersphereLoss

# 1. 실제 데이터 리스트 강제 생성 (테스트용)
# TCGA-BRCA 폴더의 파일 2개는 라벨 0, TCGA-LUAD 폴더 파일 2개는 라벨 1로 지정
base_path = '/data3/jinsol/TCGA/features_conch_256/pt_files'
brca_files = glob.glob(os.path.join(base_path, 'TCGA-BRCA/*.pt'))[:2]
luad_files = glob.glob(os.path.join(base_path, 'TCGA-LUAD/*.pt'))[:2]

data_list = [(f, 0) for f in brca_files] + [(f, 1) for f in luad_files]

# 2. 데이터로더 셋업 (WSI는 크기가 달라서 batch_size=1 필수)
dataset = TCGABags(data_list)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# 3. 네트워크 & Loss 세팅
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MonoCMIL(input_dim=512, hidden_dim=512, z_dim=256).to(device)
criterion = HypersphereLoss().to(device)

# --- 실제 학습 1-Step 시뮬레이션 ---
model.train() # 학습 모드
z_batch_list = []
label_batch_list = []

print("🔥 실제 학습 1-Step 시뮬레이션 시작...\n")

# A. 배치(총 4개 슬라이드) 통과 및 z 누적
for step, (features, label) in enumerate(dataloader):
    features = features.to(device)
    
    # 모델 포워드
    z, f_bag, A = model(features)
    
    # 텐서 누적 (z 차원: [1, 256])
    z_batch_list.append(z)
    label_batch_list.append(label.to(device))
    
    print(f"[{step+1}/4] 슬라이드 통과 완료 | 라벨: {label.item()} | 패치 개수: {features.shape[1]}")

# B. 누적된 텐서들을 하나의 배치로 병합
batch_z = torch.cat(z_batch_list, dim=0)       # [4, 256]
batch_labels = torch.cat(label_batch_list, dim=0) # [4]

# C. Loss 계산
loss, class_centers = criterion(batch_z, batch_labels)

print("\n" + "="*50)
print(f"✅ 결과: 배치 차원 {batch_z.shape}")
print(f"✅ 결과: Phase 1 Hypersphere Loss = {loss.item():.4f}")
print("="*50)